In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 52. Week 36 — Financial NLP, TF–IDF, and modality ablation

## 学習目標

- previous accessionだけを使うtext information setを監査できる
- training-only vocabulary/IDFでTF–IDFを作れる
- numeric-only、text-only、jointを同じvalidationで比較できる
- duplicate、coverage、document length、regime shiftをmetricと分けて報告できる

## 前提知識

- B5のridgeとvalidation
- Week 33–35のrepresentation
- SEC filing retrieval gate

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 52


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask

assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
assert set(fixture.partitions) == {"inner_train", "inner_validation"}

print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("numeric / sequence shape:", fixture.numeric_features.shape, fixture.token_hashes.shape)
print("locked outer rows present: False")
print("fixture hash lineage:", fixture.provenance)

fixture rows: 256
inner train / validation: 192 64
numeric / sequence shape: (256, 12) (256, 128)
locked outer rows present: False
fixture hash lineage: {'panel_artifact_sha256': '6c6008c2f28c30299e15e37613cfb0b3b22e8fd283858f5b459227c7e4a412a8', 'previous_filing_sidecar_sha256': '9ff2efef335357ff53bb1e4ba5c57f4b2e8799fc4ee5d830c55843a50026fbbc', 'normalized_manifest_sha256': '1283b9cb0992cfd2caaa942f6c869e212762c90a9abbc9a050173f5e3963daba', 'preanalysis_contract_sha256': 'fbe69fdf3b3bccba7fab70bcbb726d0df61685901cc0322d76fc66be1d7bbd6e'}


In [4]:
numeric_preprocessor = qt.fit_numeric_preprocessor(fixture.numeric_features, train_mask)
numeric_features = numeric_preprocessor.transform(fixture.numeric_features)
numeric_train = numeric_features[train_mask]
numeric_validation = numeric_features[validation_mask]
target_train = fixture.targets[train_mask]
target_validation = fixture.targets[validation_mask]
entity_validation = np.asarray(fixture.entity_ids)[validation_mask]

assert np.all(np.isfinite(numeric_features))
print("processed numeric shape:", numeric_features.shape)

processed numeric shape: (256, 24)


## 1. TF–IDF baseline

term (j)、document (i) についてsublinear TFとsmoothed IDFを

$$
\operatorname{tf}_{ij}=1+\log c_{ij},\qquad
\operatorname{idf}_{j}=\log\frac{1+n_{\mathrm{train}}}{1+\operatorname{df}_{j}}+1
$$

とし、document rowをL2 normalizeする。vocabulary rankingと (\operatorname{df}) はinner trainだけでfitする。fixtureのmany-to-one token bucketは本文を含まないがprivacy mechanismではなく、正式candidateの5,000/10,000語TF–IDFも代替しない。

In [5]:
tfidf_model = qt.fit_hashed_tfidf(
    fixture.token_hashes,
    train_mask,
    maximum_features=256,
    minimum_document_frequency=2,
)
tfidf = tfidf_model.transform(fixture.token_hashes)
assert tfidf.shape[0] == fixture.targets.size
assert np.all(np.isfinite(tfidf.data))

from scipy import sparse

numeric_ridge = qt.fit_sparse_ridge(numeric_train, target_train, ridge=1.0)
text_ridge = qt.fit_sparse_ridge(tfidf[train_mask], target_train, ridge=1.0)
joint_train = sparse.hstack([numeric_train, tfidf[train_mask]], format="csr")
joint_validation = sparse.hstack(
    [numeric_validation, tfidf[validation_mask]], format="csr"
)
joint_ridge = qt.fit_sparse_ridge(joint_train, target_train, ridge=1.0)

predictions = {
    "zero": np.zeros_like(target_validation),
    "numeric_ridge": numeric_ridge.predict(numeric_validation),
    "hashed_tfidf_ridge": text_ridge.predict(tfidf[validation_mask]),
    "joint_ridge": joint_ridge.predict(joint_validation),
}
metric_rows = [
    {"model": name, **qt.regression_error_table(target_validation, prediction, entity_validation)}
    for name, prediction in predictions.items()
]
metric_table = pd.DataFrame(metric_rows).sort_values("mae")
display(metric_table)

fig = go.Figure()
fig.add_bar(x=metric_table["model"], y=metric_table["mae"], name="row MAE")
fig.add_bar(
    x=metric_table["model"], y=metric_table["company_macro_mae"], name="company macro MAE"
)
fig.update_layout(
    title="Development-only modality ablation",
    yaxis_title="Absolute log-change error",
    barmode="group",
    template="plotly_white",
)
fig.show()

,model,mae,median_absolute_error,rmse,company_macro_mae
0,zero,0.049469,0.020475,0.114476,0.043651
2,hashed_tfidf_ridge,0.061339,0.032573,0.124007,0.053581
1,numeric_ridge,0.069652,0.027980,0.156989,0.059852
3,joint_ridge,0.074090,0.035016,0.154801,0.062959


## 2. Data-quality auditとmultimodal boundary

full retrieval gateは4,631 / 4,631 previous documents、empty 0、exact duplicate family 0、target accession leakage 0で通過した。教材fixtureは256 document hashがuniqueで、outer rowを含まない。raw/normalized SEC textとcontact-bearing User-Agentはrepository外に置く。

multimodal joint modelの改善がtext情報によるとは限らない。numeric scaling、text vocabulary、model capacity、regularization、company/date compositionを固定し、text-only / numeric-only / jointを同じrowで比較する。

In [6]:
quality_rows = pd.DataFrame(
    [
        {"check": "fixture row ids unique", "value": len(set(fixture.row_ids)), "expected": 256},
        {"check": "document hashes unique", "value": len(set(fixture.document_sha256)), "expected": 256},
        {"check": "outer rows", "value": int(np.sum(fixture.target_available_dates >= np.datetime64("2023-10-23"))), "expected": 0},
        {"check": "TF-IDF vocabulary", "value": tfidf_model.vocabulary.size, "expected": "train fitted"},
    ]
)
display(quality_rows)
assert quality_rows.loc[0, "value"] == quality_rows.loc[0, "expected"]
assert quality_rows.loc[1, "value"] == quality_rows.loc[1, "expected"]
assert quality_rows.loc[2, "value"] == 0

,check,value,expected
0,fixture row ids unique,256,256
1,document hashes unique,256,256
2,outer rows,0,0
3,TF-IDF vocabulary,256,train fitted


## 3. 失敗モード

- full corpusでvocabulary/IDFをfitする
- target accessionまたはamended future filingをfeatureへ混ぜる
- exact duplicateをcompany/time splitの両側へ置く
- missing documentをzero vectorへ黙って変換する
- TF–IDF baselineより悪いdeep modelをarchitecture名だけで採用する
- calibration intervalを同じvalidationで何度も調整する

## 4. 段階別演習

### 基礎

1. validation-only tokenがvocabularyに入らないtestを書け。
2. text coverageの分母をpanel rowで定義せよ。

### 標準

3. numeric/text/jointのparameter数とmatrix bytesを報告せよ。
4. company-macro MAEがrow MAEと異なる例を作れ。

### 研究

5. near-duplicate familyをMinHash等で監査するpre-fit protocolを書け。

## 5. Exit Criteria

- [ ] vocabulary/IDFをtrainingだけでfitした
- [ ] numeric/text/jointを同じrowで比較した
- [ ] row MAEとcompany-macro MAEを併記した
- [ ] duplicate、coverage、timestampをmodel metricから分けた
- [ ] raw textとcontact情報をrepositoryへ入れていない

## 6. 出典


- [SEC EDGAR application programming interfaces](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)
- [SEC Developer Resources](https://www.sec.gov/about/developer-resources)
- [Manning, Raghavan, and Schütze, *Introduction to Information Retrieval*](https://nlp.stanford.edu/IR-book/)